# Example notebook to run assesment of NZ challenge submission

This will run the assesement algorithms on an NZ challenge submission

#### Standard imports

In [ ]:
import tables_io, qp
import numpy as np
import os
import matplotlib.pyplot as plt
from nz_data_challenge import submit_utils, metrics, utils, evaluation
from pathlib import Path

### Paths

In [ ]:
submit_dir = 'submission/rail_knn_4tasks'
model_dir = '../models/rail_knn_4tasks'
public_dir = '../public'
results_dir = 'results/rail_knn_4tasks'
truth_dir = '../reserved'
try:
    os.makedirs(submit_dir)
except:
    pass
try:
    os.makedirs(results_dir)
except:
    pass

### Constants, grid, these should move the the library

In [ ]:
n_bins = 5
z_min = 0
z_max = 1.5
n_grid_points = 151
grid_edges = np.linspace(z_min, z_max, n_grid_points)
grid_centers = 0.5*(grid_edges[0:-1]+grid_edges[1:])
bin_edges = np.array([0., 0.32, 0.47, 0.61, 0.78, 2.5 ])
bin_centers = 0.5*(bin_edges[0:-1]+bin_edges[1:])
bin_sides = np.linspace(-0.5,n_bins-0.5,n_bins+1) 

### Filepaths, getting data

In [ ]:
taskset = 'taskset_1'
sim = 'cardinal'
scenario = '1yr'
wfd_file = f"{public_dir}/nz_challenge_{taskset}_{sim}_{scenario}_wfd.hdf5"
truth_file = f'{truth_dir}/nz_challenge_{taskset}_{sim}_{scenario}_wfd.hdf5'
nz_file = f"{submit_dir}/nz_challenge_{taskset}_{sim}_{scenario}_nz_estimate_wfd.hdf5"
bhat_file = f"{submit_dir}/nz_challenge_{taskset}_{sim}_{scenario}_bhat_wfd.hdf5"
nz_estimates = qp.read(nz_file)
test_data = tables_io.read(wfd_file)
truth = tables_io.read(truth_file)
true_redshifts = truth['redshift']
bhat_data = tables_io.read(bhat_file)
bin_assignments = np.squeeze(bhat_data['tomo_bin_index'])
hist_list = []
true_assignments = utils.get_true_bin_assignments(true_redshifts, bin_edges)

### Getting distibutions

In [ ]:
nz_distributions = utils.get_nz_distributions(nz_estimates, grid_centers, 5)
true_distributions = utils.get_true_nz_distributions(true_redshifts, bin_assignments, grid_edges, 5)

### Check that files contain what they should

In [ ]:
submit_utils.check_files(nz_file, bhat_file, wfd_file, 5)

### Compute bin assignment metrics

In [ ]:
assignment_metrics = evaluation.evaluate_bin_assignments(true_assignments, bin_assignments)
assignment_metrics

### Compute distributions metrics

In [ ]:
nz_metrics = evaluation.evaluate_distributions(true_distributions, nz_distributions, grid_edges, nz_estimates.ancil['n_objects'])
nz_metrics

### Draw the confusion matrix

In [ ]:
fig_confusion = evaluation.plot_confusion_matrix(true_assignments, bin_assignments, 5)

### Draw the n(z) distritubions

In [ ]:
fig_nz = evaluation.plot_nz_data(true_distributions, nz_distributions, grid_edges)

In [ ]:
evaluation.evaluate_submission(
    submit_dir,
    public_dir,
    truth_dir,
    results_dir,
    bin_edges,
)